In [1]:
import json

with open("../data/processed/annotated.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total data sebelum filter: {len(data)}")

# filter data dengan annotations kosong
filtered = [item for item in data if len(item["annotations"]) > 0]

print(f"Total data setelah filter: {len(filtered)}")
print(f"Data dibuang: {len(data) - len(filtered)}")

Total data sebelum filter: 1997
Total data setelah filter: 1846
Data dibuang: 151


In [2]:
def dedup_annotations(annotations):
    seen_tokens = set()
    deduped = []
    for ann in annotations:
        if ann["token"] not in seen_tokens:
            seen_tokens.add(ann["token"])
            deduped.append(ann)
    return deduped

deduped_data = []
total_removed = 0

for item in filtered:
    original_count = len(item["annotations"])
    deduped_annotations = dedup_annotations(item["annotations"])
    removed = original_count - len(deduped_annotations)
    total_removed += removed
    deduped_data.append({
        "text": item["text"],
        "annotations": deduped_annotations,
        "url": item.get("url", "")
    })

print(f"Total data: {len(deduped_data)}")
print(f"Total duplikat token yang dibuang: {total_removed}")

Total data: 1846
Total duplikat token yang dibuang: 57


In [3]:
def convert_to_bio(text, annotations):
    tokens = text.split()
    labels = ["O"] * len(tokens)

    for ann in annotations:
        ann_tokens = ann["token"].split()
        label = ann["label"]

        # cari posisi token entitas di dalam teks
        for i in range(len(tokens)):
            if tokens[i:i+len(ann_tokens)] == ann_tokens:
                labels[i] = f"B-{label}"
                for j in range(1, len(ann_tokens)):
                    labels[i+j] = f"I-{label}"


    return list(zip(tokens, labels))

bio_data = []
for item in deduped_data:
    bio = convert_to_bio(item["text"], item["annotations"])
    bio_data.append(bio)

# preview hasil konversi
print("Contoh hasil konversi BIO:")
for token, label in bio_data[0]:
    print(f"{token}\t{label}")

Contoh hasil konversi BIO:
Halo	O
Dokter,	O
saya	O
mau	O
tanya	O
soal	O
vaksin	B-TINDAKAN
BCG	I-TINDAKAN
untuk	O
anak.	O
Anak	O
saya	O
baru	O
lahir	O
beberapa	O
hari	O
lalu,	O
dan	O
saya	O
masih	O
bingung	O
kapan	O
waktu	O
yang	O
paling	O
tepat	O
untuk	O
pemberian	O
vaksin	B-TINDAKAN
BCG	I-TINDAKAN
(TBC).	O
Apakah	O
harus	O
langsung	O
setelah	O
lahir,	O
atau	O
bisa	O
ditunda?	O
Mohon	O
penjelasannya,	O
ya	O
Dok.	O


In [4]:
import random

random.seed(42)
random.shuffle(bio_data)

total = len(bio_data)
train_end = int(total * 0.7)
dev_end = int(total * 0.85)

train_data = bio_data[:train_end]
dev_data = bio_data[train_end:dev_end]
test_data = bio_data[dev_end:]

print(f"Total data: {total}")
print(f"Train: {len(train_data)}")
print(f"Dev: {len(dev_data)}")
print(f"Test: {len(test_data)}")

Total data: 1846
Train: 1292
Dev: 277
Test: 277
